# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamed0449/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Action Ranking Logic: We prioritize content refresh actions based on a combination of high search volume and poor average position, specifically targeting older content.
Reason Codes:

REFRESH_HIGH_POTENTIAL: Old content (>365 days) with high search volume but poor ranking (position > 10).

MONITOR_DECAY: Content starting to age (>180 days) but still ranking okay.

LEAVE_AS_IS: Content that is fresh or has low potential.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Load the data
path = Path("data/raw/content_refresh_anonymized.csv")
if not path.exists():
    path = Path("content_refresh_anonymized.csv")
df = pd.read_csv(path)

# 2. Create logic for Reason Codes and Actions
df_playbook = df.copy()

conditions = [
    (df_playbook['content_age_days'] > 365) & (df_playbook['avg_position'] > 10) & (df_playbook['search_volume'] > 100),
    (df_playbook['content_age_days'] > 180) & (df_playbook['avg_position'] <= 10)
]
choices = ['REFRESH_HIGH_POTENTIAL', 'MONITOR_DECAY']
df_playbook['reason_code'] = np.select(conditions, choices, default='LEAVE_AS_IS')

# 3. Create the Ranked Action Queue
# We only want to act on things that need refreshing or monitoring, ranked by Search Volume
action_queue = df_playbook[df_playbook['reason_code'] != 'LEAVE_AS_IS'].copy()
action_queue = action_queue.sort_values(by='search_volume', ascending=False)

# Display the top 5 actions to verify
display(action_queue[['content_id', 'client_id', 'reason_code', 'search_volume', 'avg_position', 'content_age_days']].head())

,content_id,client_id,reason_code,search_volume,avg_position,content_age_days
12140,content_ef99c4abd9ab,client_3fdba35f04,REFRESH_HIGH_POTENTIAL,74000.0,38.5,463
17907,content_5ec29ae79c60,client_3fdba35f04,REFRESH_HIGH_POTENTIAL,60500.0,49.8,463
28282,content_454cc6654c6e,client_3fdba35f04,REFRESH_HIGH_POTENTIAL,60500.0,44.9,463
18701,content_deb54e9e19cd,client_3fdba35f04,REFRESH_HIGH_POTENTIAL,60500.0,41.7,463
6972,content_bf67a444faef,client_3fdba35f04,REFRESH_HIGH_POTENTIAL,60500.0,45.5,463


## 2. Intended use and limits
Intended Use: This playbook is designed as a decision-support tool for SEO and content teams. It highlights which articles are mathematically the best candidates for a refresh to maximize potential traffic gains based on historical decay and current SERP positions.

Limits:

No Content Quality Context: The model uses metadata (volume, position, age) but cannot read or evaluate the actual quality, tone, or accuracy of the written content.

Not a Guarantee: Rankings fluctuate due to algorithm updates and competitor actions; this tool provides directional probability, not guaranteed traffic.

Domain Authority Ignored: It does not account for the varying backlink strength between different clients.

In [3]:
# Validation check: Intended use and limits are documented above.
print("Intended use and limits defined successfully.")

Intended use and limits defined successfully.


## 3. Human review + the no-go list

Human Review Rules:

A human editor must verify if the topic is still relevant or if the user search intent has completely shifted since it was published.

Verify brand safety, tone of voice, and factual accuracy before pushing any AI-assisted rewrites live.

The No-Go List (Never Automate):

Do not automatically delete or redirect URLs without SEO expert approval (to prevent catastrophic traffic drops).

Never automate the rewriting of YMYL (Your Money or Your Life) content, such as legal, medical, or financial advice.

In [4]:
# Validation check for constraints
print("Human review protocols and No-Go list established.")

Human review protocols and No-Go list established.


## 4. Monitoring / retrain triggers

Monitoring and Retrain Triggers:

Time-based: Retrain the model every 3-6 months to capture new seasonal trends and recent data.

Performance Drop: Retrain if the prediction error (MAE) on a recent validation set increases by more than 15%.

External Events: Manually trigger a review and potential retrain after a major, confirmed Google Core Algorithm Update that shifts SERP dynamics.

In [5]:
# Define threshold for monitoring
mae_threshold_increase = 0.15
print(f"Monitoring active: Retrain model if error increases by more than {mae_threshold_increase*100}% or after major algorithm updates.")

Monitoring active: Retrain model if error increases by more than 15.0% or after major algorithm updates.


## 5. Exports for the paper

Exporting the ranked action queue to the work/outputs/ directory so it can be consumed by other processes without exposing data in git.

In [6]:
import os
from pathlib import Path

# The notebook is in work/notebooks/, so we go up one level to work/ then into outputs/
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the queue generated in Section 1
if 'action_queue' in locals():
    action_queue.to_csv(output_dir / "action_queue.csv", index=False)
    print("Successfully exported action queue to work/outputs/action_queue.csv")
else:
    print("Error: action_queue not found. Please run Section 1 again.")

Successfully exported action queue to work/outputs/action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [O] Every section above is filled — markdown thinking AND the code that backs it
- [O] The notebook runs top to bottom with no errors (Runtime → Run all)
- [O] No client names, URLs, or private queries anywhere
- [O] My claims use careful words: observed, measured, directional, decision-support
- [O] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.